# Getting data in: uploads, volumes, read_files and COPY INTO

Analysts bring files into Databricks in three main ways. Upload a file through the UI, either to a Unity Catalog volume or straight into a managed table. Query files in place with read_files, which detects the format and schema. Load them into a table with COPY INTO, which is idempotent and skips files it already loaded. Volumes hold files; tables hold governed rows. For continuous or large-scale loading, Databricks recommends streaming tables or Auto Loader.

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end.

From the Learning Hub lesson: [Getting data in: uploads, volumes, read_files and COPY INTO](https://www.datapreplab.com/hub#topic/analyst-data-import) · Independent study resource, not affiliated with or endorsed by Databricks.

**Goal:** land two CSV files in a volume, explore them with read_files and load them with COPY INTO, twice.

**Part A: a volume and two files.** The Python cell writes two small CSV files; in real work you would upload them through the UI.

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.landing_lab;

In [ ]:
base = "/Volumes/workspace/default/landing_lab/orders/"
dbutils.fs.put(base + "orders_1.csv", "order_id,customer,amount\n1,Asha,100\n2,Ben,80\n", True)
dbutils.fs.put(base + "orders_2.csv", "order_id,customer,amount\n3,Asha,30\n4,Chen,60\n", True)

**Part B: explore in place.** Predict the inferred type of `amount` and how many rows come back, then run:

In [ ]:
%sql
SELECT *, _metadata.file_name AS file
FROM read_files('/Volumes/workspace/default/landing_lab/orders/',
                format => 'csv', header => true);

**Part C: load with COPY INTO.**

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.orders_copy_lab
  (order_id INT, customer STRING, amount DOUBLE);

In [ ]:
%sql
COPY INTO workspace.default.orders_copy_lab
FROM (SELECT CAST(order_id AS INT) AS order_id, customer, CAST(amount AS DOUBLE) AS amount
      FROM '/Volumes/workspace/default/landing_lab/orders/')
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

**Part D: run it again.** Run the same COPY INTO a second time, then add a third file and run it once more:

In [ ]:
dbutils.fs.put(base + "orders_3.csv", "order_id,customer,amount\n5,Ben,45\n", True)

In [ ]:
%sql
COPY INTO workspace.default.orders_copy_lab
FROM (SELECT CAST(order_id AS INT) AS order_id, customer, CAST(amount AS DOUBLE) AS amount
      FROM '/Volumes/workspace/default/landing_lab/orders/')
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

In [ ]:
%sql
SELECT count(*) AS rows FROM workspace.default.orders_copy_lab;

Expected evidence:

```text
Part B -> 4 rows from 2 files (orders_1.csv, orders_2.csv); amount inferred as an integer; _rescued_data is NULL on every row
Part C -> num_affected_rows 4, num_inserted_rows 4, num_skipped_corrupt_files 0 (5 if orders_3.csv is already in the volume from an earlier attempt)
Part D -> the same COPY INTO again inserts 0 rows ; after orders_3.csv it inserts 1 ; final count 5
```

**Cleanup:**

In [ ]:
%sql
DROP TABLE workspace.default.orders_copy_lab;

In [ ]:
%sql
DROP VOLUME workspace.default.landing_lab;